# DevOps Agent - Log Analysis & Incident Resolution

This agent uses **LangGraph + LangChain + OpenAI** to:
1. Query logs from **Grafana/Loki** (running in Docker)
2. Search **Runbooks** via RAG for resolution steps
3. Use **Docker CLI** and **SSH** tools for diagnostics
4. Generate an **HTML Incident Report** with errors and resolutions

## Architecture
```
Agent ──→ Runbooks RAG (FAISS + LangChain)
  │──→ Tools (Docker CLI, SSH)
  │──→ Logs (Loki/Grafana)
  └──→ HTML Summary Report
```

## Step 0: Setup

Before running this notebook:
```bash
# 1. Install dependencies
pip install -r requirements.txt

# 2. Set your OpenAI API key
export OPENAI_API_KEY="your-key-here"

# 3. Start Docker containers (Grafana + Loki)
docker-compose up -d

# 4. Run the mock log generator
python log_generator.py
```

In [ ]:
import os
import sys
from dotenv import load_dotenv

load_dotenv()

# Ensure project directory is in path
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

# Verify OpenAI API key is set
assert os.getenv("OPENAI_API_KEY"), "Please set OPENAI_API_KEY environment variable"
print("Environment ready!")

## Step 1: Start Docker Infrastructure

Start Grafana and Loki containers.

In [ ]:
# Start Docker containers
!docker-compose up -d

In [ ]:
# Verify containers are running
!docker-compose ps

In [ ]:
# Wait for Loki to be ready
import time
import requests

print("Waiting for Loki to be ready...")
for i in range(30):
    try:
        resp = requests.get("http://localhost:3100/ready", timeout=2)
        if resp.status_code == 200:
            print("Loki is ready!")
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(2)
else:
    print("Warning: Loki may not be ready yet.")

print(f"Grafana UI: http://localhost:3000 (admin/admin)")

## Step 2: Generate Mock Logs

Push realistic application logs to Loki.

In [ ]:
from log_generator import run_generator

# Generate 20 batches of 10 logs each (200 total), every 1 second
run_generator(batch_size=10, interval=1.0, total_batches=20)

## Step 3: Index Runbooks with RAG

Load runbook documents and create a FAISS vector index for retrieval.

In [ ]:
from runbook_rag import load_and_index_runbooks, query_runbooks

# Build the FAISS index from runbooks
vectorstore = load_and_index_runbooks()
print(f"\nRunbook index ready with {vectorstore.index.ntotal} vectors")

In [ ]:
# Test the RAG retrieval
result = query_runbooks("database connection failed", vectorstore)
print(result[:1000])

## Step 4: Test Individual Tools

Verify each tool works before running the full agent.

In [ ]:
from tools import query_loki_logs, query_error_logs, run_docker_command, run_ssh_command, search_runbooks

# Test: Query all logs from Loki
print("=== All Recent Logs ===")
result = query_loki_logs.invoke({"query": '{job="devops-mock-logs"}', "hours_back": 1, "limit": 10})
print(result)

In [ ]:
# Test: Query error logs only
print("=== Error Logs ===")
result = query_error_logs.invoke({"service": "", "hours_back": 1})
print(result)

In [ ]:
# Test: Docker command
print("=== Docker Containers ===")
result = run_docker_command.invoke({"command": "ps"})
print(result)

In [ ]:
# Test: SSH command (mock)
print("=== SSH: Disk Usage ===")
result = run_ssh_command.invoke({"host": "10.0.1.5", "command": "df -h"})
print(result)

In [ ]:
# Test: Runbook search
print("=== Runbook Search ===")
result = search_runbooks.invoke({"query": "container crashloopbackoff"})
print(result[:800])

## Step 5: Run the DevOps Agent

The LangGraph agent will:
1. Query Loki for ERROR/CRITICAL logs
2. Search runbooks for each error type
3. Gather diagnostic info via Docker/SSH
4. Generate an HTML incident report

In [ ]:
from agent import run_agent

# Run the agent
result = run_agent(
    task="""Analyze the recent logs from Loki for all ERROR and CRITICAL level issues.
    For each unique error type:
    1. Query the error logs from Loki
    2. Search the runbooks for resolution steps
    3. Check Docker container status
    
    Then provide a comprehensive analysis with resolutions in the required JSON format."""
)

In [ ]:
# Display the report path
report_path = result.get("report_path", "")
print(f"Report generated at: {report_path}")

# Display the last agent messages
for msg in result["messages"][-3:]:
    role = msg.__class__.__name__
    content = msg.content if hasattr(msg, 'content') else str(msg)
    if content:
        print(f"\n[{role}]")
        print(content[:2000])

In [ ]:
# Open the HTML report in browser
import webbrowser

if report_path and os.path.exists(report_path):
    webbrowser.open(f"file://{os.path.abspath(report_path)}")
    print("Report opened in browser!")
else:
    print("No report file found.")

## Step 6: Custom Queries

Run the agent with custom investigation prompts.

In [ ]:
# Investigate a specific service
result = run_agent(
    task="""Investigate the payment-service for any issues in the last hour.
    Check the logs, search runbooks for any relevant errors,
    and provide a detailed analysis with resolutions."""
)

In [ ]:
# Open latest report
report_path = result.get("report_path", "")
if report_path and os.path.exists(report_path):
    webbrowser.open(f"file://{os.path.abspath(report_path)}")
    print(f"Report: {report_path}")

## Cleanup

Stop Docker containers when done.

In [ ]:
# Stop and remove containers
# !docker-compose down -v